In [ ]:
from keras.applications import ResNet50,resnet50
from keras.src.legacy.preprocessing.image import ImageDataGenerator
from keras.optimizers import AdamW
from keras.callbacks import EarlyStopping
from keras.layers import Dense
from keras.models import Sequential,save_model
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from sklearn.model_selection import train_test_split

In [39]:
AIRPLANS_IMGS_PATH = '..\\..\\..\\..\\World Of Datasets\\Un-Structured Datasets\\Object Detection\\Natural Images\\airplane'
DRONES_IMGS_PATH = '..\\..\\..\\..\\World Of Datasets\\Un-Structured Datasets\\Object Detection\\Drones Detection Dataset\\drone_dataset\\train\\images'

# Preprocess Input Size
IMG_SIZE = 224

# Build Pre-Trained Params
WEIGHTS = 'ImageNet'
POOLING = 'avg'
TRAINABLE = False
INCLUDE_TOP = False
NUM_OF_CLASSES = 2
ACTIVATION = 'softmax'

# Flow from dataframe params
BATCH_SIZE = 64
X_COL = 'filename'
Y_COL = 'class'
CLASS_MODE = 'raw'
SEED = 42

# Training Param
EPOCHS = 20

# EarlyStopping Params
PAITENCE = 3
MONITOR = 'val_loss'

# Model Compile Params 
LOSS = 'categorical_crossentropy'
METRICS = ['accuracy']

# Optimizers Params
LR = 1e-3
WEIGHT_DECAY = 1e-3

# Test size split param
TEST_SIZE = 0.15

#Augs Params:
ROTATION = 45
BRIGHTNESS = [0.8,1.3]
HORIZ_FLIP = True

LABELS = ['Airplane','Drone']

In [ ]:
def prepare_dataset(path):
    drones_items = os.listdir(path)
    all_imgs = []
    all_labels = []
    for item in drones_items:
        img_path = os.path.join(path,item)
        all_imgs.append(img_path)
        all_labels.append(0 if 'airplane' in path else 1)
    
    return (all_imgs,all_labels)

In [ ]:
all_data = prepare_dataset(AIRPLANS_IMGS_PATH)
airplane_df = pd.DataFrame({'filename': all_data[0], 'class': all_data[1]})
airplane_df

,filename,class
0,..\..\..\..\World Of Datasets\Un-Structured Da...,0
1,..\..\..\..\World Of Datasets\Un-Structured Da...,0
2,..\..\..\..\World Of Datasets\Un-Structured Da...,0
3,..\..\..\..\World Of Datasets\Un-Structured Da...,0
4,..\..\..\..\World Of Datasets\Un-Structured Da...,0
...,...,...
722,..\..\..\..\World Of Datasets\Un-Structured Da...,0
723,..\..\..\..\World Of Datasets\Un-Structured Da...,0
724,..\..\..\..\World Of Datasets\Un-Structured Da...,0
725,..\..\..\..\World Of Datasets\Un-Structured Da...,0


In [18]:
all_data = prepare_dataset(DRONES_IMGS_PATH)
drone_df = pd.DataFrame({'filename': all_data[0], 'class': all_data[1]})
drone_df

,filename,class
0,..\..\..\..\World Of Datasets\Un-Structured Da...,1
1,..\..\..\..\World Of Datasets\Un-Structured Da...,1
2,..\..\..\..\World Of Datasets\Un-Structured Da...,1
3,..\..\..\..\World Of Datasets\Un-Structured Da...,1
4,..\..\..\..\World Of Datasets\Un-Structured Da...,1
...,...,...
1007,..\..\..\..\World Of Datasets\Un-Structured Da...,1
1008,..\..\..\..\World Of Datasets\Un-Structured Da...,1
1009,..\..\..\..\World Of Datasets\Un-Structured Da...,1
1010,..\..\..\..\World Of Datasets\Un-Structured Da...,1


In [19]:
df = pd.concat([airplane_df,drone_df],axis=0)
df

,filename,class
0,..\..\..\..\World Of Datasets\Un-Structured Da...,0
1,..\..\..\..\World Of Datasets\Un-Structured Da...,0
2,..\..\..\..\World Of Datasets\Un-Structured Da...,0
3,..\..\..\..\World Of Datasets\Un-Structured Da...,0
4,..\..\..\..\World Of Datasets\Un-Structured Da...,0
...,...,...
1007,..\..\..\..\World Of Datasets\Un-Structured Da...,1
1008,..\..\..\..\World Of Datasets\Un-Structured Da...,1
1009,..\..\..\..\World Of Datasets\Un-Structured Da...,1
1010,..\..\..\..\World Of Datasets\Un-Structured Da...,1


In [ ]:
train,test = train_test_split(df,test_size=TEST_SIZE,stratify=df['class'],random_state=SEED)
train , valid = train_test_split(train,test_size=TEST_SIZE,stratify=train['class'],random_state=SEED)

,filename,class
264,..\..\..\..\World Of Datasets\Un-Structured Da...,1
781,..\..\..\..\World Of Datasets\Un-Structured Da...,1
873,..\..\..\..\World Of Datasets\Un-Structured Da...,1
467,..\..\..\..\World Of Datasets\Un-Structured Da...,1
185,..\..\..\..\World Of Datasets\Un-Structured Da...,0
...,...,...
925,..\..\..\..\World Of Datasets\Un-Structured Da...,1
778,..\..\..\..\World Of Datasets\Un-Structured Da...,1
250,..\..\..\..\World Of Datasets\Un-Structured Da...,1
181,..\..\..\..\World Of Datasets\Un-Structured Da...,1


In [30]:
train['class'].value_counts()

class
1    731
0    525
Name: count, dtype: int64

In [31]:
valid['class'].value_counts()

class
1    129
0     93
Name: count, dtype: int64

In [32]:
test['class'].value_counts()

class
1    152
0    109
Name: count, dtype: int64

In [33]:
train

,filename,class
264,..\..\..\..\World Of Datasets\Un-Structured Da...,1
781,..\..\..\..\World Of Datasets\Un-Structured Da...,1
873,..\..\..\..\World Of Datasets\Un-Structured Da...,1
467,..\..\..\..\World Of Datasets\Un-Structured Da...,1
185,..\..\..\..\World Of Datasets\Un-Structured Da...,0
...,...,...
925,..\..\..\..\World Of Datasets\Un-Structured Da...,1
778,..\..\..\..\World Of Datasets\Un-Structured Da...,1
250,..\..\..\..\World Of Datasets\Un-Structured Da...,1
181,..\..\..\..\World Of Datasets\Un-Structured Da...,1


In [34]:
valid

,filename,class
570,..\..\..\..\World Of Datasets\Un-Structured Da...,0
998,..\..\..\..\World Of Datasets\Un-Structured Da...,1
538,..\..\..\..\World Of Datasets\Un-Structured Da...,0
552,..\..\..\..\World Of Datasets\Un-Structured Da...,1
597,..\..\..\..\World Of Datasets\Un-Structured Da...,1
...,...,...
373,..\..\..\..\World Of Datasets\Un-Structured Da...,0
186,..\..\..\..\World Of Datasets\Un-Structured Da...,0
176,..\..\..\..\World Of Datasets\Un-Structured Da...,0
83,..\..\..\..\World Of Datasets\Un-Structured Da...,0


In [35]:
test

,filename,class
273,..\..\..\..\World Of Datasets\Un-Structured Da...,0
256,..\..\..\..\World Of Datasets\Un-Structured Da...,1
762,..\..\..\..\World Of Datasets\Un-Structured Da...,1
706,..\..\..\..\World Of Datasets\Un-Structured Da...,0
156,..\..\..\..\World Of Datasets\Un-Structured Da...,0
...,...,...
784,..\..\..\..\World Of Datasets\Un-Structured Da...,1
600,..\..\..\..\World Of Datasets\Un-Structured Da...,0
520,..\..\..\..\World Of Datasets\Un-Structured Da...,0
389,..\..\..\..\World Of Datasets\Un-Structured Da...,0


In [36]:
train_gen = ImageDataGenerator(preprocessing_function=resnet50.preprocess_input,rotation_range=ROTATION,horizontal_flip=HORIZ_FLIP,brightness_range=BRIGHTNESS)

valid_gen = ImageDataGenerator(preprocessing_function=resnet50.preprocess_input)

test_gen = ImageDataGenerator(preprocessing_function=resnet50.preprocess_input)